In [5]:
import pandas as pd
import numpy as np
from collections import Counter, defaultdict

print("=" * 60)
print("HMM POS TAGGER")
print("=" * 60)

train_file = "UD_English-EWT-master/en_ewt-ud-train.conllu"
test_file = "UD_English-EWT-master/en_ewt-ud-test.conllu"


def read_conllu(file):
    sentences = []
    sentence = []

    with open(file, "r", encoding="utf-8") as f:

        for line in f:
            line = line.strip()

            if line == "":
                if sentence:
                    sentences.append(sentence)
                    sentence = []

            elif not line.startswith("#"):

                parts = line.split("\t")

                if len(parts) >= 4:

                    word = parts[1]
                    tag = parts[3]

                    if "-" not in parts[0] and "." not in parts[0]:
                        sentence.append((word, tag))

        if sentence:
            sentences.append(sentence)

    return sentences


train_data = read_conllu(train_file)
test_data = read_conllu(test_file)

print("\nDataset Loaded")
print("Training Sentences :", len(train_data))
print("Testing Sentences  :", len(test_data))


tag_counts = Counter()
word_tag_counts = defaultdict(Counter)
transition_counts = defaultdict(Counter)


for sentence in train_data:

    previous_tag = "<START>"

    for word, tag in sentence:

        tag_counts[tag] += 1
        word_tag_counts[word][tag] += 1

        transition_counts[previous_tag][tag] += 1

        previous_tag = tag

    transition_counts[previous_tag]["<END>"] += 1


transition_probability = {}

for previous_tag in transition_counts:

    total = sum(transition_counts[previous_tag].values())

    transition_probability[previous_tag] = {}

    for tag in transition_counts[previous_tag]:

        transition_probability[previous_tag][tag] = (
            transition_counts[previous_tag][tag] / total
        )


emission_probability = {}

for word in word_tag_counts:

    total = sum(word_tag_counts[word].values())

    emission_probability[word] = {}

    for tag in word_tag_counts[word]:

        emission_probability[word][tag] = (
            word_tag_counts[word][tag] / total
        )


print("\nHMM Matrices Created")
print("Number of POS Tags :", len(tag_counts))
print("Number of Words    :", len(word_tag_counts))


def predict(sentence):

    predicted_tags = []

    previous_tag = "<START>"

    for word in sentence:

        if word in emission_probability:

            possible_tags = emission_probability[word]

            best_tag = None
            best_score = -1

            for tag in possible_tags:

                emission = emission_probability[word][tag]

                transition = transition_probability.get(
                    previous_tag, {}
                ).get(tag, 0)

                score = emission * transition

                if score > best_score:

                    best_score = score
                    best_tag = tag

            if best_tag is None:

                best_tag = max(
                    possible_tags,
                    key=possible_tags.get
                )

        else:

            best_tag = max(
                tag_counts,
                key=tag_counts.get
            )

        predicted_tags.append(best_tag)
        previous_tag = best_tag

    return predicted_tags


actual_tags = []
predicted_tags = []


for sentence in test_data:

    words = [word for word, tag in sentence]
    tags = [tag for word, tag in sentence]

    predictions = predict(words)

    actual_tags.extend(tags)
    predicted_tags.extend(predictions)


actual_tags = np.array(actual_tags)
predicted_tags = np.array(predicted_tags)


accuracy = np.mean(actual_tags == predicted_tags)


print("\n" + "=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)

print(f"\nAccuracy : {accuracy:.4f}")
print(f"Accuracy Percentage : {accuracy * 100:.2f}%")


print("\n" + "=" * 60)
print("USER INPUT POS TAGGING")
print("=" * 60)

sentence = input("\nEnter a sentence: ")

words = sentence.split()

sample_prediction = predict(words)


print("\nPOS TAGGING RESULT")

for word, tag in zip(words, sample_prediction):

    print(f"{word:15} -> {tag}")


print("\n" + "=" * 60)
print("HMM POS TAGGING COMPLETED")
print("=" * 60)

HMM POS TAGGER

Dataset Loaded
Training Sentences : 12544
Testing Sentences  : 2077

HMM Matrices Created
Number of POS Tags : 17
Number of Words    : 19674

EVALUATION RESULTS

Accuracy : 0.8536
Accuracy Percentage : 85.36%

USER INPUT POS TAGGING



Enter a sentence:  The student is reading a book



POS TAGGING RESULT
The             -> DET
student         -> NOUN
is              -> AUX
reading         -> VERB
a               -> DET
book            -> NOUN

HMM POS TAGGING COMPLETED
